In [1]:
import subprocess
import os

# Check hostname and environment
print("=== Hostname ===")
print(subprocess.run(["hostname"], capture_output=True, text=True).stdout.strip())

print("\n=== GPU Info (nvidia-smi) ===")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("nvidia-smi not available:", result.stderr)

print("\n=== GPU Count ===")
result2 = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True
)
if result2.returncode == 0:
    gpus = result2.stdout.strip().split("\n")
    print(f"Number of GPUs detected: {len(gpus)}")
    for i, gpu in enumerate(gpus):
        print(f"  GPU {i}: {gpu}")
else:
    print("Could not query GPU details")

=== Hostname ===
jupyter-yniu2

=== GPU Info (nvidia-smi) ===
Thu Mar 12 17:40:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:07:00.0 Off |                    0 |
| N/A   27C    P0             54W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |          

In [2]:
import torch

print("=== PyTorch CUDA Info ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs (torch): {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {props.name}, {props.total_memory / 1024**3:.1f} GB")
        if "A100" in props.name:
            print(f"    -> A100 confirmed!")
else:
    print("No CUDA devices found via PyTorch")

ModuleNotFoundError: No module named 'torch'

In [3]:
import subprocess

print("=== Kubernetes / Pod Environment Check ===")
# Check if running inside a k8s pod
k8s_indicators = [
    ("KUBERNETES_SERVICE_HOST", os.environ.get("KUBERNETES_SERVICE_HOST")),
    ("KUBERNETES_PORT", os.environ.get("KUBERNETES_PORT")),
    ("POD_NAME", os.environ.get("POD_NAME")),
    ("HOSTNAME", os.environ.get("HOSTNAME")),
]
for key, val in k8s_indicators:
    print(f"  {key}: {val}")

# Check /var/run/secrets/kubernetes.io presence (standard k8s pod indicator)
k8s_secret_path = "/var/run/secrets/kubernetes.io"
print(f"\n  k8s service account path exists: {os.path.exists(k8s_secret_path)}")

print("\n=== CPU / Memory Info ===")
cpu_info = subprocess.run(["nproc"], capture_output=True, text=True)
print(f"CPU cores: {cpu_info.stdout.strip()}")
mem_info = subprocess.run(["free", "-h"], capture_output=True, text=True)
print(mem_info.stdout)

=== Kubernetes / Pod Environment Check ===
  KUBERNETES_SERVICE_HOST: 10.96.0.1
  KUBERNETES_PORT: tcp://10.96.0.1:443
  POD_NAME: None
  HOSTNAME: jupyter-yniu2

  k8s service account path exists: False

=== CPU / Memory Info ===
CPU cores: 256
               total        used        free      shared  buff/cache   available
Mem:           1.0Ti        56Gi       423Gi       6.4Gi       528Gi       940Gi
Swap:             0B          0B          0B



In [4]:
import subprocess, sys

print("=== vLLM Installation Check ===")

# Method 1: try importing vllm
try:
    import vllm
    print(f"vLLM is installed: version {vllm.__version__}")
except ImportError as e:
    print(f"vLLM is NOT importable: {e}")

# Method 2: pip show
result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "vllm"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("\n--- pip show vllm ---")
    print(result.stdout)
else:
    print("\nvllm not found via pip show")

# Method 3: check pip list for related packages
result2 = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=columns"],
    capture_output=True, text=True
)
vllm_related = [line for line in result2.stdout.splitlines() if "vllm" in line.lower()]
print("\n--- pip list (vllm-related) ---")
if vllm_related:
    for line in vllm_related:
        print(line)
else:
    print("No vllm-related packages found in pip list")

=== vLLM Installation Check ===
vLLM is NOT importable: No module named 'vllm'

vllm not found via pip show

--- pip list (vllm-related) ---
No vllm-related packages found in pip list
